# Philo Inference Pipeline

**Purpose:** Load new data → Add features → Load model → Evaluate(Recall@10)



In [9]:
NEW_DATA_PATH = '../data/playback_sessions.parquet'

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import pickle

from utils.train_test_split import chronological_train_test_split, create_ranking_test_set
from utils.evaluation import evaluate_model

print('Setup complete!')

Setup complete!


In [10]:
# Configuration
MODEL_PATH = '../models/philo_lgb_recommender.pkl'
print(f'Data: {NEW_DATA_PATH}')
print(f'Model: {MODEL_PATH}')


Data: ../data/playback_sessions.parquet
Model: ../models/philo_lgb_recommender.pkl


In [3]:
# Load data
print('Loading data...')
df = pd.read_parquet(NEW_DATA_PATH)
print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst few rows:')
df.head()

Loading data...
Loaded 12,941,230 rows
Columns: ['user_id', 'playback_session_id', 'show_id', 'asset_type', 'episode_id', 'day', 'time', 'watch_minutes']

First few rows:


,user_id,playback_session_id,show_id,asset_type,episode_id,day,time,watch_minutes
0,1,1,1,VOD,1,03,20:16,54
1,1,2,1,VOD,2,03,11:05,53
2,1,3,1,VOD,3,03,22:46,44
3,1,4,2,VOD,4,01,22:56,1
4,1,5,3,RECORDING,5,06,15:26,27


In [4]:
# Feature engineering - Create timestamps
print('\nFeature Engineering:')
print('='*70)

print('\nSample data:')
print('  Day values:', df['day'].head(3).tolist())
print('  Time values:', df['time'].head(3).tolist())

# The 'day' column contains just day-of-month (e.g., '03', '15', '27')
# We need to add year and month to create proper timestamps
# Assuming January 2024 - adjust if your data is from a different period
YEAR = '2024'
MONTH = '01'

print(f'\nUsing: {YEAR}-{MONTH} as base date')
print('Note: Change YEAR/MONTH if your data is from a different period\n')

# Create full timestamp: YYYY-MM-DD HH:MM
df['timestamp'] = pd.to_datetime(
    YEAR + '-' + MONTH + '-' + df['day'].str.zfill(2) + ' ' + df['time'],
    format='%Y-%m-%d %H:%M',
    errors='coerce'
)

# Check for failed conversions
null_count = df['timestamp'].isnull().sum()
if null_count > 0:
    print(f'⚠️  {null_count:,} / {len(df):,} timestamps failed to parse')
    print('Removing invalid timestamps...')
    df = df[df['timestamp'].notnull()].copy()
else:
    print(f'✓ All {len(df):,} timestamps parsed successfully!')

# Sort by timestamp
df = df.sort_values('timestamp').reset_index(drop=True)

# Create implicit interaction score
max_watch = df['watch_minutes'].max()
df['implicit_interaction_score'] = (df['watch_minutes'] / max_watch) * 100

print(f'\n✓ Features created successfully!')
print(f'  Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'  Duration: {(df["timestamp"].max() - df["timestamp"].min()).days} days')
print(f'  Rows: {len(df):,}')
print(f'  Users: {df["user_id"].nunique():,}')
print(f'  Shows: {df["show_id"].nunique():,}')


Feature Engineering:

Sample data:
  Day values: ['03', '03', '03']
  Time values: ['20:16', '11:05', '22:46']

Using: 2024-01 as base date
Note: Change YEAR/MONTH if your data is from a different period

✓ All 12,941,230 timestamps parsed successfully!

✓ Features created successfully!
  Date range: 2024-01-01 00:00:00 to 2024-01-30 23:59:00
  Duration: 29 days
  Rows: 12,941,230
  Users: 100,000
  Shows: 13,349


In [5]:
# Prepare data for evaluation (use ALL data - no train/test split)
print('Preparing data for evaluation...')
print(f'  Total rows: {len(df):,}')
print(f'  Unique users: {df["user_id"].nunique():,}')
print(f'  Unique shows: {df["show_id"].nunique():,}')

# Create ground truth from ALL data
# This will evaluate how well the model recommends shows that users actually watched
test_ground_truth = df.groupby('user_id').agg({
    'show_id': lambda x: set(x),
    'watch_minutes': 'sum'
}).reset_index()

# Expand to one row per user-show pair
ground_truth_rows = []
for _, row in test_ground_truth.iterrows():
    for show_id in row['show_id']:
        ground_truth_rows.append({
            'user_id': row['user_id'],
            'show_id': show_id,
            'relevance_score': 1  # All watched shows are relevant
        })

test_ground_truth = pd.DataFrame(ground_truth_rows)

# Create empty train_seen_dict (no filtering - we want to see all recommendations)
train_seen_dict = {}

print(f'\nGround truth created:')
print(f'  Users: {test_ground_truth["user_id"].nunique():,}')
print(f'  Total user-show pairs: {len(test_ground_truth):,}')
print(f'  Avg shows per user: {len(test_ground_truth) / test_ground_truth["user_id"].nunique():.1f}')

Preparing data for evaluation...
  Total rows: 12,941,230
  Unique users: 100,000
  Unique shows: 13,349

Ground truth created:
  Users: 100,000
  Total user-show pairs: 2,239,763
  Avg shows per user: 22.4


In [6]:
!ls ../models

__init__.py                        philo_als_recommender.pkl
als_lgb_ensemble.py                philo_lgb_recommender.pkl
als_recommender.py                 popularity_lgb_recommender.py
lgb_ranker.py                      popularity_recommender.py
philo_als_plus_lgb_recommender.pkl


In [7]:
# Load model
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

print(f'Model loaded: {type(model).__name__}')

Model loaded: LGBRanker


In [8]:
# Evaluate - Get Recall@10
print('Evaluating model...')

metrics = evaluate_model(
    model=model,
    test_ground_truth=test_ground_truth,
    train_seen_dict=train_seen_dict,
    user_col='user_id',
    item_col='show_id',
    top_n=10,
    verbose=True,
    batch_size=1000
)

print('\n' + '='*70)
print(f"Recall@10:    {metrics['recall@10']*100:.2f}%")
print(f"MRR@10:       {metrics['mrr@10']:.4f}")
print(f"Precision@10: {metrics['precision@10']*100:.2f}%")
print('='*70)

Evaluating model...
Evaluating model on 100,000 users...
  Processing batch 1/100 (0/100,000 users)...
  Processing batch 11/100 (10,000/100,000 users)...
  Processing batch 21/100 (20,000/100,000 users)...
  Processing batch 31/100 (30,000/100,000 users)...
  Processing batch 41/100 (40,000/100,000 users)...
  Processing batch 51/100 (50,000/100,000 users)...
  Processing batch 61/100 (60,000/100,000 users)...
  Processing batch 71/100 (70,000/100,000 users)...
  Processing batch 81/100 (80,000/100,000 users)...
  Processing batch 91/100 (90,000/100,000 users)...

  Evaluation Results (Top-10)
  Users Evaluated: 100,000
  Recall@10:    21.18%
  MRR@10:       0.3544
  Precision@10: 14.24%


Recall@10:    21.18%
MRR@10:       0.3544
Precision@10: 14.24%
